# 08 - Efficiency Benchmark
Parameters, **FLOPs counted by `torch.utils.flop_counter`** (replaces the hand-written conv-only counter), median / p95 latency with warm-up, CUDA-event timing on GPU, batch 1 and 16, FP32 and FP16.

Internal-consistency checks are **asserted**: throughput must equal `batch * 1000 / median latency` (the submitted Table II listed a 16.22 ms latency next to a throughput that corresponds to 18.22 ms).

In [2]:
import os, sys, json, warnings, importlib
PROJECT_ROOT = os.path.abspath("..")
if not os.path.isdir(os.path.join(PROJECT_ROOT, "notebooks")):
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
# Prefer a configured data root only when it actually contains the paired folders.
def _has_paired_data(root):
    if not root:
        return False
    return ((os.path.isdir(os.path.join(root, "train", "train", "GT")) and
             os.path.isdir(os.path.join(root, "train", "train", "NoisyLR"))) or
            (os.path.isdir(os.path.join(root, "GT")) and
             os.path.isdir(os.path.join(root, "NoisyLR"))))
if not _has_paired_data(os.environ.get("SEMICON_DATA")):
    os.environ["SEMICON_DATA"] = PROJECT_ROOT
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C = importlib.reload(C)
# Print explicit results and data roots below
from utils.experiment import build_model, CONFIGS
from utils.analysis import count_flops, latency_ms
dev = C.device()
RUNS = os.path.join(PROJECT_ROOT, "results_kaggle")
DATA = PROJECT_ROOT
C.OUT = RUNS
C.RUNS = os.path.join(RUNS, "runs")
C.TABLES = os.path.join(RUNS, "tables")
C.FIGS = os.path.join(RUNS, "figures")
C.DATA_ROOT = DATA
print("RUNS =", RUNS)
print("DATA =", DATA)
names = ["baseline", "film_control", "full", "NAFNet-small", "Restormer-small", "SwinIR-small", "NAFNet-base", "Restormer-base", "SwinIR-base"]
n_run, n_warm = (3, 1) if C.MODE == "smoke" else ((30, 5) if dev == "cuda" else (5, 1))
H = 64 if C.MODE == "smoke" else 128
print(f"hardware={dev}; timing repetitions={n_run}, warmups={n_warm} (CPU defaults are shortened to keep the benchmark responsive)")
rows = []
for c in names:
    print(f"Benchmarking {c}...", flush=True)
    m = build_model(CONFIGS[c]); p = sum(x.numel() for x in m.parameters())
    r = dict(config=c, params=p, GFLOPs=round(count_flops(m, (1, 1, H, H)), 3))
    for b in (1, 16):
        cpu_runs = max(5, n_run // 5) if C.MODE != "smoke" and b == 16 else max(2, n_run // (5 if b == 16 else 1))
        x = latency_ms(m, (b, 1, H, H), "cpu", n_warm=n_warm, n_run=cpu_runs)
        assert abs(x["throughput_img_s"] - b * 1000 / x["median_ms"]) < 1e-6
        r[f"cpu_b{b}_ms"], r[f"cpu_b{b}_img_s"] = round(x["median_ms"], 2), round(x["throughput_img_s"], 1)
        if dev == "cuda":
            for half in (False, True):
                g = latency_ms(m, (b, 1, H, H), "cuda", half=half, n_warm=n_warm, n_run=n_run)
                r[f"gpu_b{b}_{'fp16' if half else 'fp32'}_ms"], r[f"gpu_b{b}_{'fp16' if half else 'fp32'}_img_s"] = round(g["median_ms"], 2), round(g["throughput_img_s"], 1)
    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats(); m.cuda().eval()(torch.randn(1, 1, H, H, device="cuda")); r["gpu_peak_MB"] = round(torch.cuda.max_memory_allocated() / 2**20, 1)
    rows.append(r)
    pd.DataFrame(rows).to_csv(os.path.join(C.TABLES, "table_efficiency.csv"), index=False)
    print(f"Finished {c}.", flush=True)
bench = pd.DataFrame(rows); print(bench.to_string(index=False)); bench.to_csv(os.path.join(C.TABLES, "table_efficiency.csv"), index=False)
print("\nhardware:", torch.cuda.get_device_name(0) if dev == "cuda" else "CPU only", "| input", (1, 1, H, H), "| median over", n_run, "runs after", n_warm, "warm-ups")

RUNS = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_kaggle
DATA = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow
hardware=cpu; timing repetitions=5, warmups=1 (CPU defaults are shortened to keep the benchmark responsive)
Benchmarking baseline...
Finished baseline.
Benchmarking film_control...
Finished film_control.
Benchmarking full...
Finished full.
Benchmarking NAFNet-small...
Finished NAFNet-small.
Benchmarking Restormer-small...
Finished Restormer-small.
Benchmarking SwinIR-small...
Finished SwinIR-small.
Benchmarking NAFNet-base...
Finished NAFNet-base.
Benchmarking Restormer-base...
Finished Restormer-base.
Benchmarking SwinIR-base...
Finished SwinIR-base.
         config  params  GFLOPs  cpu_b1_ms  cpu_b1_img_s  cpu_b16_ms  cpu_b16_img_s
       baseline   86945   3.898     267.09           3.7     4932.81            3.2
   film_control  116138   4.844     410.31           2.4     6459.21            2.5
           full  1161